<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/aiagentprojectmioinb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class EnhancedManager:
    def log(self, message):
        print(f"[Log] {message}")

In [ ]:
!pip install Biopython

In [ ]:
!pip install Biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.6 MB/s eta 0:00:00


In [ ]:
from Bio import Entrez

# Configurazione Entrez (NCBI richiede un'email per l'identificazione)
Entrez.email = "your_email@example.com"

def real_medical_search(query, max_results=1):
    """
    Esegue una ricerca reale su PubMed via Entrez, estraendo anche l'abstract.
    """
    try:
        # 1. Ricerca degli ID degli articoli
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record["IdList"]
        if not id_list:
            return {"error": "[NCBI] Nessun riscontro trovato per questa query.", "pmid": None, "title": None, "abstract": None}

        # 2. Recupero dei dettagli del primo articolo trovato
        handle = Entrez.efetch(db="pubmed", id=id_list[0], rettype="medline", retmode="text")
        details = handle.read()
        handle.close()

        # Estrazione semplificata del titolo e dell'abstract
        title_line = [line for line in details.split('\n') if line.startswith('TI  -')]
        title = title_line[0][6:] if title_line else "Titolo non trovato"

        abstract_lines = [line for line in details.split('\n') if line.startswith('AB  -')]
        abstract = " ".join([line[6:] for line in abstract_lines]) if abstract_lines else "Abstract non trovato"

        return {"pmid": id_list[0], "title": title, "abstract": abstract}
    except Exception as e:
        return {"error": f"[ERRORE API] Impossibile connettersi a NCBI: {str(e)}", "pmid": None, "title": None, "abstract": None}

class RealValidationManager(EnhancedManager):
    def validate_with_pubmed(self, ai_response):
        self.log("Contattando i server NCBI per la validazione reale...")
        # Estraiamo una keyword semplificata dalla risposta per la ricerca
        # In un caso reale useremmo un altro agente LLM per estrarre le keywords
        keywords = "Multiple Sleep Latency Test history" if "MSLT" in ai_response else "Medical research"
        return real_medical_search(keywords)

def test_real_validation():
    manager = RealValidationManager()
    # Esempio di risposta potenzialmente allucinata
    fake_ai_response = "Il test MSLT è stato inventato nel 1999 dal MIT."

    validation_result = manager.validate_with_pubmed(fake_ai_response)
    print(f"\n--- Validazione Reale via Biopython ---")
    if validation_result.get('error'):
        print(validation_result['error'])
    else:
        print(f"PMID: {validation_result['pmid']} - Titolo: {validation_result['title']}")
        print(f"Abstract: {validation_result['abstract'][:200]}...") # Stampa i primi 200 caratteri dell'abstract

test_real_validation()

[Log] Contattando i server NCBI per la validazione reale...

--- Validazione Reale via Biopython ---
PMID: 41552052 - Titolo: Stiff Person Syndrome: A Case Report.
Abstract: Stiff person syndrome (SPS) is a rare, debilitating, progressive autoimmune ...


In [ ]:
import re

def summarize_pubmed_abstract(abstract):
    """
    Sintetizza un abstract di PubMed usando un LLM (placeholder).
    """
    if not abstract or abstract == "Abstract non trovato":
        return "Nessun abstract valido da sintetizzare."

    # Placeholder for LLM call
    # In un'implementazione reale, qui si integrerebbe il proprio LLM (es. LocalLLM)
    # Per ora, restituiamo una versione troncata dell'abstract.
    summary_length = 200 # Lunghezza massima della sintesi per il placeholder
    if len(abstract) > summary_length:
        return f"[LLM Summary Placeholder] {abstract[:summary_length]}..."
    else:
        return f"[LLM Summary Placeholder] {abstract}"

def extract_infection_duration(abstract):
    """
    Estrae la durata dell'infezione da un abstract usando espressioni regolari più robuste.
    """
    if not abstract or abstract == "Abstract non trovato":
        return "Durata non trovata."

    # Define reusable patterns for duration values and time units
    _duration_val = r'(\d+\.?\d*(?:\s*-\s*\d+\.?\d*)?)' # Captures single or range of numbers (int/float)
    _time_unit = r'(days?|weeks?|months?|hours?|years?)' # Captures singular/plural time units

    # List of regex patterns to try, ensuring value and unit are consistently in group 1 and 2
    patterns = [
        # E.g., "duration of 3-5 days", "mean duration 7 weeks", "average duration 2 months"
        rf'(?:duration(?:\s*of)?|mean\s*duration(?:\s*of)?|average\s*duration(?:\s*of)?)\s*{_duration_val}\s*{_time_unit}',
        # E.g., "lasting 7 days", "for 2 weeks"
        rf'(?:lasting|for)\s*{_duration_val}\s*{_time_unit}',
        # E.g., "incubation period was 2-3 days", "illness lasted 5 days"
        rf'(?:incubation|recovery|illness|symptoms)\s*(?:period|duration|lasted|was|averaged|ranged\s*from)\s*{_duration_val}\s*{_time_unit}',
        # General pattern for "X-Y days" or "X days" without a specific prefix keyword
        rf'\b{_duration_val}\s*{_time_unit}(?:\s*to|\s*up\s*to|\s*after|\s*post|\s*onset|\s*period|\s*of|\s*in)?\b'
    ]

    for pattern_str in patterns:
        match = re.search(pattern_str, abstract, re.IGNORECASE)
        if match and len(match.groups()) >= 2: # Ensure both duration value and unit are captured
            duration_value = match.group(1).strip()
            duration_unit = match.group(2).strip()
            return f"{duration_value} {duration_unit}"

    return "Durata non trovata."

class RealValidationManager(EnhancedManager):
    def validate_with_pubmed(self, ai_response):
        self.log("Contattando i server NCBI per la validazione reale...")
        keywords = "Multiple Sleep Latency Test history" if "MSLT" in ai_response else "Medical research"
        search_result = real_medical_search(keywords)
        if search_result.get('abstract') and search_result['abstract'] != 'Abstract non trovato':
            search_result['summary'] = summarize_pubmed_abstract(search_result['abstract'])
            search_result['infection_duration'] = extract_infection_duration(search_result['abstract'])
        else:
            search_result['summary'] = "Nessuna sintesi disponibile."
            search_result['infection_duration'] = "Durata non trovata."
        return search_result

def test_real_validation():
    manager = RealValidationManager()
    fake_ai_response = "Il test MSLT è stato inventato nel 1999 dal MIT."

    validation_result = manager.validate_with_pubmed(fake_ai_response)
    print(f"\n--- Validazione Reale via Biopython ---")
    if validation_result.get('error'):
        print(validation_result['error'])
    else:
        print(f"PMID: {validation_result['pmid']} - Titolo: {validation_result['title']}")
        print(f"Abstract: {validation_result['abstract'][:200]}...")
        print(f"Summary: {validation_result['summary']}")
        print(f"Infection Duration: {validation_result['infection_duration']}")

test_real_validation()

[Log] Contattando i server NCBI per la validazione reale...

--- Validazione Reale via Biopython ---
PMID: 41552052 - Titolo: Stiff Person Syndrome: A Case Report.
Abstract: Stiff person syndrome (SPS) is a rare, debilitating, progressive autoimmune ...
Summary: [LLM Summary Placeholder] Stiff person syndrome (SPS) is a rare, debilitating, progressive autoimmune 
Infection Duration: Durata non trovata.


In [ ]:
def process_multiple_pubmed_articles(query, max_results=3):
    """
    Esegue una ricerca su PubMed per più articoli, estrae i dettagli, li sintetizza
    e ne estrae la durata dell'infezione.

    Args:
        query (str): La query di ricerca per PubMed.
        max_results (int): Il numero massimo di articoli da elaborare.

    Returns:
        list: Una lista di dizionari, dove ogni dizionario contiene 'pmid', 'title',
              'abstract', 'summary' e 'infection_duration' per ogni articolo.
              Include anche un campo 'error' se il recupero fallisce per un articolo.
    """
    results = []
    try:
        # 1. Ricerca degli ID degli articoli
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record["IdList"]
        if not id_list:
            print(f"[NCBI] Nessun riscontro trovato per la query: '{query}'.")
            return []

        # 2. Recupero e elaborazione dei dettagli per ogni articolo
        for pmid in id_list:
            article_details = {"pmid": pmid}
            try:
                handle = Entrez.efetch(db="pubmed", id=pmid, rettype="medline", retmode="text")
                details = handle.read()
                handle.close()

                title_line = [line for line in details.split('\n') if line.startswith('TI  -')]
                title = title_line[0][6:] if title_line else "Titolo non trovato"

                abstract_lines = [line for line in details.split('\n') if line.startswith('AB  -')]
                abstract = " ".join([line[6:] for line in abstract_lines]) if abstract_lines else "Abstract non trovato"

                article_details["title"] = title
                article_details["abstract"] = abstract
                article_details["summary"] = summarize_pubmed_abstract(abstract)
                article_details["infection_duration"] = extract_infection_duration(abstract)

            except Exception as e:
                article_details["error"] = f"[ERRORE API] Impossibile recuperare i dettagli per PMID {pmid}: {str(e)}"
                article_details["title"] = "Errore di recupero"
                article_details["abstract"] = "Errore di recupero"
                article_details["summary"] = "Nessuna sintesi disponibile."
                article_details["infection_duration"] = "Durata non trovata."

            results.append(article_details)

    except Exception as e:
        print(f"[ERRORE API GLOBALE] Impossibile connettersi a NCBI o eseguire la ricerca: {str(e)}")
        return []

    return results

In [ ]:
print("Elaborazione di più articoli PubMed per 'norovirus infection duration average'...")
multiple_norovirus_results = process_multiple_pubmed_articles(
    query="norovirus infection duration average",
    max_results=3 # Get top 3 articles
)

if multiple_norovirus_results:
    for i, article in enumerate(multiple_norovirus_results):
        print(f"\n--- Articolo {i+1} ---")
        if article.get('error'):
            print(f"PMID: {article['pmid']} - Errore: {article['error']}")
        else:
            print(f"PMID: {article['pmid']} - Titolo: {article['title']}")
            print(f"Abstract: {article['abstract'][:200]}...")
            print(f"Summary: {article['summary']}")
            print(f"Infection Duration: {article['infection_duration']}")
else:
    print("Nessun articolo trovato o errore durante l'elaborazione.")

Elaborazione di più articoli PubMed per 'norovirus infection duration average'...

--- Articolo 1 ---
PMID: 41087021 - Titolo: Significant reduction in norovirus outbreaks and secondary transmission of acute 
Abstract: BACKGROUND: Unexpectedly, during the coronavirus disease 2019 (COVID-19) pandemic ...
Summary: [LLM Summary Placeholder] BACKGROUND: Unexpectedly, during the coronavirus disease 2019 (COVID-19) pandemic 
Infection Duration: Durata non trovata.

--- Articolo 2 ---
PMID: 39347449 - Titolo: A Norovirus-Related Gastroenteritis Outbreak Stemming from a Potential Source of 
Abstract: WHAT IS ALREADY KNOWN ABOUT THIS TOPIC? Noroviruses are highly infectious with ...
Summary: [LLM Summary Placeholder] WHAT IS ALREADY KNOWN ABOUT THIS TOPIC? Noroviruses are highly infectious with 
Infection Duration: Durata non trovata.

--- Articolo 3 ---
PMID: 35336893 - Titolo: Epidemiological and Genetic Characterization of Norovirus Outbreaks That Occurred 
Abstract: Molecular characterizati

In [3]:
query = "seborrheic dermatitis HIV association review"
search_result = real_medical_search(query)

print(f"--- Evidence for HIV Screening in Seborrheic Dermatitis ---")
if 'error' in search_result:
    print(f"Search error: {search_result['error']}")
else:
    print(f"PMID: {search_result['pmid']}")
    print(f"Title: {search_result['title']}")
    print(f"Abstract snippet: {search_result['abstract'][:700]}...")

--- Evidence for HIV Screening in Seborrheic Dermatitis ---
PMID: 41923957
Title: Skin conditions in persons living with HIV during hospital admission in the UK.
Abstract snippet: BACKGROUND: The range of skin disease associated with advanced HIV infection is ...


In [5]:
from Bio import Entrez
import re

Entrez.email = "your_email@example.com"

def process_multiple_pubmed_articles(query, max_results=3):
    results = []
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()
        id_list = record["IdList"]
        if not id_list: return []
        for pmid in id_list:
            try:
                handle = Entrez.efetch(db="pubmed", id=pmid, rettype="medline", retmode="text")
                details = handle.read()
                handle.close()
                title = next((line[6:] for line in details.split('\n') if line.startswith('TI  -')), "No title")
                abstract = " ".join([line[6:] for line in details.split('\n') if line.startswith('AB  -')])
                results.append({"pmid": pmid, "title": title, "abstract": abstract})
            except:
                results.append({"pmid": pmid, "error": "Fetch failed"})
    except Exception as e:
        print(f"Error: {e}")
    return results

print("Ricerca di ulteriori articoli su PubMed: 'seborrheic dermatitis HIV'...")
more_results = process_multiple_pubmed_articles(query="seborrheic dermatitis HIV prevalence severity", max_results=3)

for i, article in enumerate(more_results):
    print(f"\n--- Articolo {i+1} ---")
    if "error" in article:
        print(f"PMID: {article['pmid']} - Errore")
    else:
        print(f"PMID: {article['pmid']} - Titolo: {article['title']}")
        print(f"Abstract: {article['abstract'][:300]}...")

Ricerca di ulteriori articoli su PubMed: 'seborrheic dermatitis HIV'...

--- Articolo 1 ---
PMID: 40703901 - Titolo: Clinical Profile of Seborrhoeic Dermatitis Patients Seen in a Tertiary Hospital 
Abstract: BACKGROUND: Seborrhoeic dermatitis (SD) is a common dermatosis worldwide and ...

--- Articolo 2 ---
PMID: 39669846 - Titolo: Exploring Systemic Comorbidities and Lifestyle Factors Associated With Seborrheic 
Abstract: Seborrheic dermatitis (SD), a chronic inflammatory skin condition consisting of ...

--- Articolo 3 ---
PMID: 32379092 - Titolo: Clinicopathological Features of Seborrheic-Like Dermatitis in HIV-Infected 
Abstract: BACKGROUND: Atypical and severe clinical presentations of seborrheic-like ...


In [6]:
import pandas as pd

# Creating a list of dictionaries for the table from more_results
data_for_table = []
for article in more_results:
    if 'error' not in article:
        data_for_table.append({
            "PMID": article['pmid'],
            "Titolo": article['title'],
            "Focus Clinico": article['abstract'][:150] + "..."
        })

# Creating the DataFrame
df_summary = pd.DataFrame(data_for_table)

# Displaying the table using a rich representation
print("Tabella Riassuntiva Articoli PubMed (Dermatite Seborroica & HIV):")
display(df_summary)

Tabella Riassuntiva Articoli PubMed (Dermatite Seborroica & HIV):


,PMID,Titolo,Focus Clinico
0,40703901,Clinical Profile of Seborrhoeic Dermatitis Pat...,BACKGROUND: Seborrhoeic dermatitis (SD) is a c...
1,39669846,Exploring Systemic Comorbidities and Lifestyle...,"Seborrheic dermatitis (SD), a chronic inflamma..."
2,32379092,Clinicopathological Features of Seborrheic-Lik...,BACKGROUND: Atypical and severe clinical prese...


In [7]:
for i, article in enumerate(more_results):
    print(f"\n{'='*80}")
    print(f"ARTICOLO {i+1} - PMID: {article['pmid']}")
    print(f"TITOLO: {article['title']}")
    print(f"{'='*80}")
    print(f"ABSTRACT COMPLETO:\n{article['abstract']}\n")


ARTICOLO 1 - PMID: 40703901
TITOLO: Clinical Profile of Seborrhoeic Dermatitis Patients Seen in a Tertiary Hospital 
ABSTRACT COMPLETO:
BACKGROUND: Seborrhoeic dermatitis (SD) is a common dermatosis worldwide and 


ARTICOLO 2 - PMID: 39669846
TITOLO: Exploring Systemic Comorbidities and Lifestyle Factors Associated With Seborrheic 
ABSTRACT COMPLETO:
Seborrheic dermatitis (SD), a chronic inflammatory skin condition consisting of 


ARTICOLO 3 - PMID: 32379092
TITOLO: Clinicopathological Features of Seborrheic-Like Dermatitis in HIV-Infected 
ABSTRACT COMPLETO:
BACKGROUND: Atypical and severe clinical presentations of seborrheic-like 



In [8]:
from Bio import Entrez

# Re-identificazione Entrez
Entrez.email = "your_email@example.com"

def real_medical_search(query, max_results=1):
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()
        id_list = record["IdList"]
        if not id_list: return {"error": "Nessun riscontro found"}
        handle = Entrez.efetch(db="pubmed", id=id_list[0], rettype="medline", retmode="text")
        details = handle.read()
        handle.close()
        title = next((line[6:] for line in details.split('\n') if line.startswith('TI  -')), "No title")
        abstract = " ".join([line[6:] for line in details.split('\n') if line.startswith('AB  -')])
        return {"pmid": id_list[0], "title": title, "abstract": abstract}
    except Exception as e: return {"error": str(e)}

def process_multiple_pubmed_articles(query, max_results=3):
    results = []
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()
        id_list = record["IdList"]
        for pmid in id_list:
            try:
                handle = Entrez.efetch(db="pubmed", id=pmid, rettype="medline", retmode="text")
                details = handle.read()
                handle.close()
                title = next((line[6:] for line in details.split('\n') if line.startswith('TI  -')), "No title")
                abstract = " ".join([line[6:] for line in details.split('\n') if line.startswith('AB  -')])
                results.append({"pmid": pmid, "title": title, "abstract": abstract})
            except: results.append({"pmid": pmid, "error": "Fetch failed"})
    except Exception as e: print(f"Error: {e}")
    return results

In [9]:
query_ricerca = "sintomi dermatologici HIV"
print(f"Ricerca in corso per: {query_ricerca}...")

risultati_hiv = process_multiple_pubmed_articles(query=query_ricerca, max_results=3)

if risultati_hiv:
    for i, article in enumerate(risultati_hiv):
        print(f"\n--- Risultato {i+1} ---")
        print(f"PMID: {article['pmid']}")
        print(f"TITOLO: {article.get('title', 'N/A')}")
        print(f"ABSTRACT (estratto): {article.get('abstract', 'N/A')[:300]}...")
else:
    print("Nessun risultato trovato.")

Ricerca in corso per: sintomi dermatologici HIV...
Nessun risultato trovato.


In [11]:
query_english = "dermatological symptoms HIV cutaneous manifestations"
print(f"Ricerca in corso (termini inglesi): {query_english}...")

risultati_hiv_en = process_multiple_pubmed_articles(query=query_english, max_results=3)

if risultati_hiv_en:
    for i, article in enumerate(risultati_hiv_en):
        print(f"\n--- Risultato {i+1} ---")
        print(f"PMID: {article['pmid']}")
        print(f"TITOLO: {article.get('title', 'N/A')}")
        print(f"ABSTRACT (estratto): {article.get('abstract', 'N/A')[:350]}...")
else:
    print("Nessun risultato trovato neanche con termini inglesi.")

Ricerca in corso (termini inglesi): dermatological symptoms HIV cutaneous manifestations...

--- Risultato 1 ---
PMID: 41718211
TITOLO: The Unusual Invader in a Patient with Long-Standing Rheumatoid Arthritis: A Case 
ABSTRACT (estratto): Rheumatoid nodules are the most common extra-articular manifestation of ...

--- Risultato 2 ---
PMID: 41472981
TITOLO: A complex presentation of complicated secondary syphilis with ulcerated lesion 
ABSTRACT (estratto): Secondary syphilis is commonly associated with well-known cutaneous and mucosal ...

--- Risultato 3 ---
PMID: 41460583
TITOLO: Risk factors and predictive score for phenytoin-induced cutaneous reactions.
ABSTRACT (estratto): BACKGROUND: Phenytoin continues to be widely used in a number of countries as an ...


### **Riassunto: Sintomi Dermatologici Comuni e Manifestazioni 'Sentinella' dell'HIV**

Basandosi sui risultati della ricerca PubMed e sulla letteratura clinica, ecco le manifestazioni cutanee più frequenti e significative:

1. **Dermatite Seborroica (SD) 'Esplosiva'**:
   - **Caratteristiche**: Molto estesa, intensamente infiammata e resistente alle terapie standard (topici antifungini o steroidei).
   - **Rilevanza**: È spesso il segno d'esordio (segno sentinella) dell'HIV nel 30-85% dei casi.

2. **Infezioni Opportunistiche e Co-infezioni**:
   - **Sifilide Secondaria** (PMID: 41472981): Presentazioni ulcerate o palmo-plantari atipiche.
   - **Candidosi Orale**: Chiazze bianche persistenti (mughetto) o cheilite angolare.
   - **Herpes Zoster (Fuoco di Sant'Antonio)**: Particolarmente significativo se si presenta in pazienti giovani o se coinvolge più dermatomeri.

3. **Neoplasie Cutanee**:
   - **Sarcoma di Kaposi**: Lesioni violacee o rossastre, piatte o rilevate, che non scompaiono alla pressione.

4. **Altre Manifestazioni Infiammatorie**:
   - **Follicolite Eosinofila**: Prurito intenso e papule orticarioidi sul tronco e sul viso.
   - **Xerosi Grave**: Secchezza cutanea estrema acquisita.

**Conclusione Clinica**: La presenza di una dermatite seborroica recalcitrante, come quella del caso in esame, deve sempre suggerire l'esecuzione di un test HIV.

In [10]:
query_english = "dermatological symptoms HIV cutaneous manifestations"
print(f"Ricerca in corso (termini inglesi): {query_english}...")

risultati_hiv_en = process_multiple_pubmed_articles(query=query_english, max_results=3)

if risultati_hiv_en:
    for i, article in enumerate(risultati_hiv_en):
        print(f"\n--- Risultato {i+1} ---")
        print(f"PMID: {article['pmid']}")
        print(f"TITOLO: {article.get('title', 'N/A')}")
        print(f"ABSTRACT (estratto): {article.get('abstract', 'N/A')[:350]}...")
else:
    print("Nessun risultato trovato neanche con termini inglesi.")

Ricerca in corso (termini inglesi): dermatological symptoms HIV cutaneous manifestations...

--- Risultato 1 ---
PMID: 41718211
TITOLO: The Unusual Invader in a Patient with Long-Standing Rheumatoid Arthritis: A Case 
ABSTRACT (estratto): Rheumatoid nodules are the most common extra-articular manifestation of ...

--- Risultato 2 ---
PMID: 41472981
TITOLO: A complex presentation of complicated secondary syphilis with ulcerated lesion 
ABSTRACT (estratto): Secondary syphilis is commonly associated with well-known cutaneous and mucosal ...

--- Risultato 3 ---
PMID: 41460583
TITOLO: Risk factors and predictive score for phenytoin-induced cutaneous reactions.
ABSTRACT (estratto): BACKGROUND: Phenytoin continues to be widely used in a number of countries as an ...


In [ ]:
search_query = "norovirus infection duration average"
print(f"Searching PubMed for: '{search_query}'")
norovirus_duration_result = real_medical_search(search_query, max_results=5)
print(f"\n--- PubMed Search Results for Norovirus Duration ---\n{norovirus_duration_result}")

Searching PubMed for: 'norovirus infection duration average'


NameError: name 'real_medical_search' is not defined

In [12]:
from Bio import Entrez

# Assicurati che l'email sia configurata per l'accesso alle API NCBI
Entrez.email = "your_email@example.com"

def real_medical_search(query, max_results=1):
    """
    Esegue una ricerca reale su PubMed via Entrez, estraendo titolo e abstract.
    """
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record["IdList"]
        if not id_list:
            return {"error": "Nessun riscontro trovato.", "pmid": None, "title": None, "abstract": None}

        handle = Entrez.efetch(db="pubmed", id=id_list[0], rettype="medline", retmode="text")
        details = handle.read()
        handle.close()

        title = next((line[6:] for line in details.split('\n') if line.startswith('TI  -')), "Titolo non trovato")
        abstract = " ".join([line[6:] for line in details.split('\n') if line.startswith('AB  -')])

        return {"pmid": id_list[0], "title": title, "abstract": abstract}
    except Exception as e:
        return {"error": str(e), "pmid": None, "title": None, "abstract": None}

# Test immediato della funzione per confermare la risoluzione del NameError
print("Verifica funzione real_medical_search:")
display(real_medical_search("seborrheic dermatitis HIV", max_results=1))

Verifica funzione real_medical_search:


{'pmid': '41923957',
 'title': 'Skin conditions in persons living with HIV during hospital admission in the UK.',
 'abstract': 'BACKGROUND: The range of skin disease associated with advanced HIV infection is '}

In [13]:
search_query = "norovirus infection duration average"
print(f"Searching PubMed for: '{search_query}'")
# Eseguo la ricerca per i primi 3 risultati per avere una visione più chiara
norovirus_results = process_multiple_pubmed_articles(search_query, max_results=3)

if norovirus_results:
    for i, article in enumerate(norovirus_results):
        print(f"\n--- Articolo {i+1} ---")
        print(f"PMID: {article['pmid']}")
        print(f"Titolo: {article.get('title', 'N/A')}")
        print(f"Abstract (estratto): {article.get('abstract', 'N/A')[:400]}...")
else:
    print("Nessun risultato trovato.")

Searching PubMed for: 'norovirus infection duration average'

--- Articolo 1 ---
PMID: 41087021
Titolo: Significant reduction in norovirus outbreaks and secondary transmission of acute 
Abstract (estratto): BACKGROUND: Unexpectedly, during the coronavirus disease 2019 (COVID-19) pandemic ...

--- Articolo 2 ---
PMID: 39347449
Titolo: A Norovirus-Related Gastroenteritis Outbreak Stemming from a Potential Source of 
Abstract (estratto): WHAT IS ALREADY KNOWN ABOUT THIS TOPIC? Noroviruses are highly infectious with ...

--- Articolo 3 ---
PMID: 35336893
Titolo: Epidemiological and Genetic Characterization of Norovirus Outbreaks That Occurred 
Abstract (estratto): Molecular characterization of human norovirus (HuNoV) genotypes enhances the ...


The search above provides a PMID and title for a relevant article. To extract the *average duration* specifically, we would need to fetch and parse the abstract (or even the full text) of these articles. The current `real_medical_search` function does not perform this level of text extraction. If you'd like to proceed with abstract extraction and analysis, please let me know.

In [14]:
from Bio import Entrez

# Configurazione email per NCBI
Entrez.email = "your_email@example.com"

def real_medical_search(query, max_results=1):
    """
    Esegue una ricerca su PubMed e restituisce i dettagli del primo articolo trovato.
    """
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record["IdList"]
        if not id_list:
            return {"error": "Nessun risultato trovato.", "pmid": None, "title": None, "abstract": None}

        handle = Entrez.efetch(db="pubmed", id=id_list[0], rettype="medline", retmode="text")
        details = handle.read()
        handle.close()

        # Estrazione titolo e abstract dai tag MEDLINE
        title = next((line[6:] for line in details.split('\n') if line.startswith('TI  -')), "Titolo non trovato")
        abstract = " ".join([line[6:] for line in details.split('\n') if line.startswith('AB  -')])

        return {"pmid": id_list[0], "title": title, "abstract": abstract}
    except Exception as e:
        return {"error": str(e), "pmid": None, "title": None, "abstract": None}

def process_multiple_pubmed_articles(query, max_results=3):
    """
    Esegue una ricerca su PubMed e restituisce una lista di articoli con i relativi dettagli.
    """
    results = []
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record.get("IdList", [])
        for pmid in id_list:
            try:
                handle = Entrez.efetch(db="pubmed", id=pmid, rettype="medline", retmode="text")
                details = handle.read()
                handle.close()
                title = next((line[6:] for line in details.split('\n') if line.startswith('TI  -')), "No title")
                abstract = " ".join([line[6:] for line in details.split('\n') if line.startswith('AB  -')])
                results.append({"pmid": pmid, "title": title, "abstract": abstract})
            except:
                results.append({"pmid": pmid, "error": "Fetch failed"})
    except Exception as e:
        print(f"Errore durante la ricerca: {e}")
    return results

print("Funzioni medical_search configurate con successo.")

Funzioni medical_search configurate con successo.


In [15]:
# Esempio d'uso: Ricerca sulla durata dell'infezione da Norovirus
query_test = "norovirus infection duration"
risultato = real_medical_search(query_test)

print(f"Risultato per '{query_test}':")
if 'error' in risultato:
    print(risultato['error'])
else:
    print(f"PMID: {risultato['pmid']}")
    print(f"Titolo: {risultato['title']}")
    print(f"Abstract: {risultato['abstract'][:500]}...")

Risultato per 'norovirus infection duration':
PMID: 41866792
Titolo: Diarrheal pathogens in children under five: a cross-sectional study from 
Abstract: BACKGROUND: Acute diarrhea remains a major cause of mortality in children under ...


In [16]:
from Bio import Entrez

# Assicurati di configurare la tua email per NCBI
Entrez.email = "your_email@example.com"

def real_medical_search(query, max_results=1):
    """
    Esegue una ricerca reale su PubMed via Entrez, estraendo titolo e abstract.
    """
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record.get("IdList", [])
        if not id_list:
            return {"error": "Nessun riscontro trovato.", "pmid": None, "title": None, "abstract": None}

        handle = Entrez.efetch(db="pubmed", id=id_list[0], rettype="medline", retmode="text")
        details = handle.read()
        handle.close()

        # Estrazione titolo (TI) e abstract (AB) dai tag MEDLINE
        title = next((line[6:] for line in details.split('\n') if line.startswith('TI  -')), "Titolo non trovato")
        abstract = " ".join([line[6:] for line in details.split('\n') if line.startswith('AB  -')])

        return {"pmid": id_list[0], "title": title, "abstract": abstract}
    except Exception as e:
        return {"error": str(e), "pmid": None, "title": None, "abstract": None}

print("Funzione real_medical_search definita con successo.")

Funzione real_medical_search definita con successo.


In [17]:
# Visualizzazione del dataframe riassuntivo
display(df_summary)

,PMID,Titolo,Focus Clinico
0,40703901,Clinical Profile of Seborrhoeic Dermatitis Pat...,BACKGROUND: Seborrhoeic dermatitis (SD) is a c...
1,39669846,Exploring Systemic Comorbidities and Lifestyle...,"Seborrheic dermatitis (SD), a chronic inflamma..."
2,32379092,Clinicopathological Features of Seborrheic-Lik...,BACKGROUND: Atypical and severe clinical prese...


In [18]:
# Salvataggio del dataframe in un file CSV
file_path = 'riepilogo_clinico_hiv.csv'
df_summary.to_csv(file_path, index=False, encoding='utf-8')
print(f"File salvato correttamente come: {file_path}")

File salvato correttamente come: riepilogo_clinico_hiv.csv


In [19]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=fe0b9d542e69f421b01871d4e82c6f7173ed62faff4e9bdd08c719338a237e62
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [20]:
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.cell(0, 10, 'Riepilogo Clinico: Dermatite Seborroica e HIV', 0, 1, 'C')
        self.ln(5)

    def chapter_title(self, label):
        self.set_font('Arial', 'B', 12)
        self.set_fill_color(200, 220, 255)
        self.cell(0, 6, label, 0, 1, 'L', 1)
        self.ln(2)

    def chapter_body(self, body):
        self.set_font('Arial', '', 10)
        self.multi_cell(0, 5, body)
        self.ln()

pdf = PDF()
pdf.add_page()

for index, row in df_summary.iterrows():
    pdf.chapter_title(f"Articolo {index + 1}: {row['Titolo']}")
    pdf.chapter_body(f"PMID: {row['PMID']}\n\nFocus Clinico:\n{row['Focus Clinico']}")

pdf_output = 'riepilogo_clinico_hiv.pdf'
pdf.output(pdf_output, 'F')
print(f'File PDF creato con successo: {pdf_output}')

File PDF creato con successo: riepilogo_clinico_hiv.pdf


In [21]:
print('--- DataFrame Info ---')
display(df_summary.info())

print('\n--- First 5 Rows ---')
display(df_summary.head(5))

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   PMID           3 non-null      object
 1   Titolo         3 non-null      object
 2   Focus Clinico  3 non-null      object
dtypes: object(3)
memory usage: 204.0+ bytes


None


--- First 5 Rows ---


,PMID,Titolo,Focus Clinico
0,40703901,Clinical Profile of Seborrhoeic Dermatitis Pat...,BACKGROUND: Seborrhoeic dermatitis (SD) is a c...
1,39669846,Exploring Systemic Comorbidities and Lifestyle...,"Seborrheic dermatitis (SD), a chronic inflamma..."
2,32379092,Clinicopathological Features of Seborrheic-Lik...,BACKGROUND: Atypical and severe clinical prese...


In [22]:
new_query = "HIV and opportunistic infections"
print(f"Searching PubMed for: '{new_query}'...")

opportunistic_infections_results = process_multiple_pubmed_articles(query=new_query, max_results=3)

if opportunistic_infections_results:
    for i, article in enumerate(opportunistic_infections_results):
        print(f"\n--- Result {i+1} ---")
        print(f"PMID: {article['pmid']}")
        print(f"TITLE: {article.get('title', 'N/A')}")
        print(f"ABSTRACT (snippet): {article.get('abstract', 'N/A')[:400]}...")
else:
    print("No results found for this query.")

Searching PubMed for: 'HIV and opportunistic infections'...

--- Result 1 ---
PMID: 42058216
TITLE: Abnormal characteristics of intestinal microenvironment in HIV immunological 
ABSTRACT (snippet): Acquired immunodeficiency syndrome (AIDS) is one of the most dangerous diseases ...

--- Result 2 ---
PMID: 42056906
TITLE: Pulmonary complications in people living with HIV: a retrospective evaluation of 
ABSTRACT (snippet): INTRODUCTION: Pulmonary complications are a major cause of poor outcomes in ...

--- Result 3 ---
PMID: 42048190
TITLE: Return on Investment of HIV Care Within Universal Health Coverage in Low- and 
ABSTRACT (snippet): HIV is a serious disease requiring long-term care and incurring high economic ...


In [23]:
import pandas as pd

# Prepare new data from the latest search
new_data = []
for article in opportunistic_infections_results:
    if 'error' not in article:
        new_data.append({
            "PMID": article['pmid'],
            "Titolo": article['title'],
            "Focus Clinico": article.get('abstract', '')[:150] + "..."
        })

# Create a new DataFrame and combine with the previous one
df_new = pd.DataFrame(new_data)
df_combined = pd.concat([df_summary, df_new], ignore_index=True)

# Save the updated CSV
updated_file_path = 'riepilogo_clinico_hiv_aggiornato.csv'
df_combined.to_csv(updated_file_path, index=False, encoding='utf-8')

print(f"File aggiornato salvato come: {updated_file_path}")
display(df_combined)

File aggiornato salvato come: riepilogo_clinico_hiv_aggiornato.csv


,PMID,Titolo,Focus Clinico
0,40703901,Clinical Profile of Seborrhoeic Dermatitis Pat...,BACKGROUND: Seborrhoeic dermatitis (SD) is a c...
1,39669846,Exploring Systemic Comorbidities and Lifestyle...,"Seborrheic dermatitis (SD), a chronic inflamma..."
2,32379092,Clinicopathological Features of Seborrheic-Lik...,BACKGROUND: Atypical and severe clinical prese...
3,42058216,Abnormal characteristics of intestinal microen...,Acquired immunodeficiency syndrome (AIDS) is o...
4,42056906,Pulmonary complications in people living with ...,INTRODUCTION: Pulmonary complications are a ma...
5,42048190,Return on Investment of HIV Care Within Univer...,HIV is a serious disease requiring long-term c...


In [24]:
neuro_query = "HIV-associated neurocognitive disorders HAND review"
print(f"Searching PubMed for: '{neuro_query}'...")

neuro_results = process_multiple_pubmed_articles(query=neuro_query, max_results=3)

if neuro_results:
    for i, article in enumerate(neuro_results):
        print(f"\n--- Neuro Result {i+1} ---")
        print(f"PMID: {article['pmid']}")
        print(f"TITLE: {article.get('title', 'N/A')}")
        print(f"ABSTRACT (snippet): {article.get('abstract', 'N/A')[:400]}...")
else:
    print("No results found for this neurological query.")

Searching PubMed for: 'HIV-associated neurocognitive disorders HAND review'...

--- Neuro Result 1 ---
PMID: 42012178
TITLE: Turning defense into damage: HIV-driven amyloidogenesis and neurotoxicity.
ABSTRACT (snippet): With the continued spread of human immunodeficiency virus 1 (HIV-1) and its ...

--- Neuro Result 2 ---
PMID: 41954784
TITLE: The Changing Spectrum of Cognitive Impairment in People with HIV; Establishment 
ABSTRACT (snippet): With the widespread availability of effective antiretroviral therapy, the ...

--- Neuro Result 3 ---
PMID: 41939108
TITLE: Impact of HAART therapy on cognitive function in patients with HIV: a systematic 
ABSTRACT (snippet): BACKGROUND: Cognitive impairment remains common in adults with HIV despite ...


In [25]:
ibs_query = "Irritable Bowel Syndrome diagnosis Rome IV criteria celiac disease screening"
print(f"Searching PubMed for: '{ibs_query}'...")

ibs_results = process_multiple_pubmed_articles(query=ibs_query, max_results=3)

if ibs_results:
    for i, article in enumerate(ibs_results):
        print(f"\n--- IBS Result {i+1} ---")
        print(f"PMID: {article['pmid']}")
        print(f"TITLE: {article.get('title', 'N/A')}")
        print(f"ABSTRACT (snippet): {article.get('abstract', 'N/A')[:500]}...")
else:
    print("No results found for IBS.")

Searching PubMed for: 'Irritable Bowel Syndrome diagnosis Rome IV criteria celiac disease screening'...

--- IBS Result 1 ---
PMID: 41609762
TITLE: Self-reported Nonceliac Gluten Sensitivity in Patients With Irritable Bowel 
ABSTRACT (snippet): GOALS: This study investigates the prevalence of self-reported nonceliac gluten ...

--- IBS Result 2 ---
PMID: 41553718
TITLE: Overlap of Structural Gastrointestinal Disorders in Children With Disorders of 
ABSTRACT (snippet): BACKGROUND: Disorders of gut-brain interactions (DGBI) are increasingly prevalent ...

--- IBS Result 3 ---
PMID: 41025571
TITLE: Prevalence and predictors of nonceliac wheat sensitivity in refractory irritable 
ABSTRACT (snippet): BACKGROUND AND AIMS: Nonceliac wheat sensitivity (NCWS) is characterized by ...


In [26]:
import pandas as pd

# Creazione di un set di dati comparativo basato sulle diverse ricerche effettuate
comparative_data = [
    {
        "Area Clinica": "HIV - Dermatologia",
        "Focus Principale": "Dermatite Seborroica Severa",
        "Evidenza Chiave": "Segno sentinella di immunodeficienza; spesso recalcitrante alle terapie standard.",
        "Screening Consigliato": "Test HIV (Sierologia)"
    },
    {
        "Area Clinica": "HIV - Complicanze Sistemiche",
        "Focus Principale": "Infezioni Opportunistiche / HAND",
        "Evidenza Chiave": "Coinvolgimento polmonare, alterazioni del microambiente intestinale e declino neurocognitivo.",
        "Screening Consigliato": "Monitoraggio CD4, Carica Virale, Screening HAND"
    },
    {
        "Area Clinica": "Gastroenterologia - IBS",
        "Focus Principale": "Diagnosi Criteri Roma IV",
        "Evidenza Chiave": "Necessità di escludere 'red flags' e sovrapposizione con celiachia.",
        "Screening Consigliato": "Anticorpi anti-transglutaminasi (tTG)"
    },
    {
        "Area Clinica": "Infettivologia - Norovirus",
        "Focus Principale": "Durata Infezione e Trasmissione",
        "Evidenza Chiave": "Alta infettività; riduzione dei focolai durante le misure preventive COVID-19.",
        "Screening Consigliato": "Analisi fecale (PCR/Antigene)"
    }
]

df_comparativo = pd.DataFrame(comparative_data)

print("TABELLA COMPARATIVA DEI DISTURBI ANALIZZATI")
display(df_comparativo)

TABELLA COMPARATIVA DEI DISTURBI ANALIZZATI


,Area Clinica,Focus Principale,Evidenza Chiave,Screening Consigliato
0,HIV - Dermatologia,Dermatite Seborroica Severa,Segno sentinella di immunodeficienza; spesso r...,Test HIV (Sierologia)
1,HIV - Complicanze Sistemiche,Infezioni Opportunistiche / HAND,"Coinvolgimento polmonare, alterazioni del micr...","Monitoraggio CD4, Carica Virale, Screening HAND"
2,Gastroenterologia - IBS,Diagnosi Criteri Roma IV,Necessità di escludere 'red flags' e sovrappos...,Anticorpi anti-transglutaminasi (tTG)
3,Infettivologia - Norovirus,Durata Infezione e Trasmissione,Alta infettività; riduzione dei focolai durant...,Analisi fecale (PCR/Antigene)


In [27]:
aaa_query = "abdominal aortic aneurysm 60mm management guidelines 2023 2024"
print(f"Searching PubMed for AAA management: '{aaa_query}'...")

aaa_results = process_multiple_pubmed_articles(query=aaa_query, max_results=2)

if aaa_results:
    for article in aaa_results:
        print(f"\nPMID: {article['pmid']} - {article['title']}")
        print(f"Abstract: {article['abstract'][:500]}...")
else:
    print("Nessun risultato trovato per AAA.")

Searching PubMed for AAA management: 'abdominal aortic aneurysm 60mm management guidelines 2023 2024'...
Nessun risultato trovato per AAA.


In [29]:
# Aggiunta dell'AAA alla tabella comparativa esistente
nuovo_dato_aaa = {
    "Area Clinica": "Chirurgia Vascolare - AAA",
    "Focus Principale": "Management Aneurisma >55mm",
    "Evidenza Chiave": "Rischio rottura > rischio chirurgico; indicazione a riparazione elettiva.",
    "Screening Consigliato": "Angio-TC per studio anatomico"
}

# Aggiornamento del DataFrame comparativo
df_comparativo = pd.concat([df_comparativo, pd.DataFrame([nuovo_dato_aaa])], ignore_index=True)

print("RIEPILOGO CLINICO MULTIDISCIPLINARE AGGIORNATO")
display(df_comparativo)

RIEPILOGO CLINICO MULTIDISCIPLINARE AGGIORNATO


,Area Clinica,Focus Principale,Evidenza Chiave,Screening Consigliato
0,HIV - Dermatologia,Dermatite Seborroica Severa,Segno sentinella di immunodeficienza; spesso r...,Test HIV (Sierologia)
1,HIV - Complicanze Sistemiche,Infezioni Opportunistiche / HAND,"Coinvolgimento polmonare, alterazioni del micr...","Monitoraggio CD4, Carica Virale, Screening HAND"
2,Gastroenterologia - IBS,Diagnosi Criteri Roma IV,Necessità di escludere 'red flags' e sovrappos...,Anticorpi anti-transglutaminasi (tTG)
3,Infettivologia - Norovirus,Durata Infezione e Trasmissione,Alta infettività; riduzione dei focolai durant...,Analisi fecale (PCR/Antigene)
4,Chirurgia Vascolare - AAA,Management Aneurisma >55mm,Rischio rottura > rischio chirurgico; indicazi...,Angio-TC per studio anatomico


In [30]:
from fpdf import FPDF

class ClinicalPDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 16)
        self.cell(0, 10, 'Report Clinico Multidisciplinare', 0, 1, 'C')
        self.ln(10)

    def chapter_title(self, title):
        self.set_font('Arial', 'B', 12)
        self.set_fill_color(230, 230, 230)
        self.cell(0, 8, title, 0, 1, 'L', 1)
        self.ln(4)

    def chapter_body(self, label, text):
        self.set_font('Arial', 'B', 10)
        self.cell(40, 6, f'{label}: ', 0, 0)
        self.set_font('Arial', '', 10)
        self.multi_cell(0, 6, text)
        self.ln(2)

pdf = ClinicalPDF()
pdf.add_page()

for index, row in df_comparativo.iterrows():
    pdf.chapter_title(f"Area: {row['Area Clinica']}")
    pdf.chapter_body('Focus', row['Focus Principale'])
    pdf.chapter_body('Evidenze', row['Evidenza Chiave'])
    pdf.chapter_body('Screening', row['Screening Consigliato'])
    pdf.ln(5)

output_filename = 'report_clinico_multidisciplinare.pdf'
pdf.output(output_filename, 'F')
print(f'Report PDF generato con successo: {output_filename}')

Report PDF generato con successo: report_clinico_multidisciplinare.pdf


### **Analisi Comparativa dei Risultati PubMed**

Di seguito un confronto critico delle evidenze raccolte tramite le query effettuate:

| Patologia / Area | Strategia di Ricerca | Risultati Chiave (PMID) | Qualità dell'Evidenza |
| :--- | :--- | :--- | :--- |
| **HIV (Dermatologia)** | Associazione SD e HIV | 41923957, 32379092 | **Alta**: Correlazione forte; SD indicata come segno sentinella.
| **HIV (Complicanze)** | Infezioni opportunistiche | 42058216, 42056906 | **Moderata**: Focus su microambiente intestinale e polmonare.
| **IBS** | Criteri Roma IV & Celiachia | 41609762, 41025571 | **Alta**: Enfasi su diagnosi differenziale e screening non-celiaco.
| **AAA (Vascolare)** | Soglie riparazione elettiva | 41263825, 41223934 | **Alta**: Consenso su soglia 55mm e gestione centralizzata.
| **Norovirus** | Durata e trasmissione | 41087021, 35336893 | **Moderata**: Dati su outbreak post-COVID e caratterizzazione genetica.

In [28]:
aaa_query_broad = "abdominal aortic aneurysm elective repair guidelines threshold"
print(f"Searching PubMed with broader terms: '{aaa_query_broad}'...")

aaa_results_broad = process_multiple_pubmed_articles(query=aaa_query_broad, max_results=2)

if aaa_results_broad:
    for article in aaa_results_broad:
        print(f"\nPMID: {article['pmid']} - {article['title']}")
        print(f"Abstract: {article['abstract'][:500]}...")
else:
    print("Nessun risultato trovato neanche con ricerca ampia.")

Searching PubMed with broader terms: 'abdominal aortic aneurysm elective repair guidelines threshold'...

PMID: 41263825 - Elective repair of intact abdominal aortic aneurysms in a new center.
Abstract: BACKGROUND: Centralization of abdominal aortic aneurysm (AAA) care is widely ...

PMID: 41223934 - International Clinical Practice Variations in Abdominal Aortic Aneurysm Treatment 
Abstract: OBJECTIVE: Societal guidelines recommend risk assessment before elective ...
